# EA2 – Ideación, diseño y documentación de la necesidad

**Proyecto Integrador III** · Braian Rico · Samuel Rosero · Diego Cuervo

En este notebook obtenemos un dataset de Hugging Face por código, lo cargamos en un `DataFrame` de pandas, hacemos una revisión descriptiva y generamos automáticamente un resumen `.txt` en `EA2/resultados/`.

## 1. Necesidad analítica (escenario hipotético)

> **Aviso:** el escenario es **hipotético**, planteado con fines académicos. No existe un cliente real.

El equipo de operaciones de un sistema público de bicicletas compartidas necesita saber **cómo cambia la demanda por hora** según la franja horaria, el tipo de día (laboral, no laboral o festivo), la estación del año y el clima, y **en qué se diferencian los usuarios casuales de los registrados**. Con eso podría planificar la redistribución de bicicletas y el mantenimiento.

**Pregunta descriptiva:** ¿Cómo varía la cantidad de alquileres por hora según la franja horaria, el tipo de día, la estación y las condiciones del clima, y qué diferencias hay entre usuarios casuales y registrados durante 2011 y 2012?

**Unidad de análisis:** cada fila representa **una hora de servicio** del sistema, con la cantidad de alquileres de esa hora y las condiciones de calendario y clima.

## 2. Dataset elegido

| Dato | Valor |
|---|---|
| Dataset en Hugging Face | [`t22000t/bike-sharing-tabular`](https://huggingface.co/datasets/t22000t/bike-sharing-tabular) |
| Split | `train` |
| Revisión (commit) | `6b645e0cedbc5208fcca11ac94544b088478bf61` |
| Fuente original | [Bike Sharing Dataset – UCI Machine Learning Repository (id 275)](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset) |
| Autores originales | Hadi Fanaee-T y João Gama |
| Licencia de la fuente | CC BY 4.0 |
| Fecha de consulta | 2026-10-25 |

El dataset de Hugging Face es una copia tabular del archivo **por hora** del dataset de UCI: datos del sistema *Capital Bikeshare* (Washington D.C.) entre 2011 y 2012. Fijamos la **revisión** para que cualquiera obtenga exactamente los mismos datos, aunque el repositorio cambie después.

## 3. Preparación del entorno

Buscamos la raíz del repositorio (la carpeta que contiene `src/clases_po`) y agregamos `src` a la ruta de Python para poder importar nuestras clases.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "clases_po").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

import pandas as pd

from clases_po import HFDatasetLoader, Reportes

## 4. Obtención del dataset y carga en un DataFrame

Usamos la clase `HFDatasetLoader`. Cargamos el dataset **completo** (sin `max_rows` y sin *streaming*).

In [ ]:
DATASET = "t22000t/bike-sharing-tabular"
CONFIG = None
SPLIT = "train"
REVISION = "6b645e0cedbc5208fcca11ac94544b088478bf61"

loader = HFDatasetLoader(
    dataset_name=DATASET,
    config=CONFIG,
    split=SPLIT,
    revision=REVISION,
)
df = loader.load()
loader.info()

## 5. Revisión descriptiva del DataFrame

### 5.1 Algunos registros

In [ ]:
df.head(10)

### 5.2 Filas y columnas

In [ ]:
filas, columnas = df.shape
print(f"Filas: {filas:,}".replace(",", "."))
print(f"Columnas: {columnas}")

### 5.3 Nombres de columnas y tipos de dato

In [ ]:
df.dtypes.rename("tipo").to_frame()

### 5.4 Valores nulos por columna

In [ ]:
df.isna().sum().rename("nulos").to_frame()

### 5.5 Estadísticas básicas de las columnas numéricas

In [ ]:
df.describe().round(3).T

## 6. ¿Qué representa cada fila y qué variables sirven para el análisis?

Cada fila es **una hora de servicio** del sistema entre el 1 de enero de 2011 y el 31 de diciembre de 2012.

### Diccionario de variables

Basado **solo** en la documentación oficial de UCI (id 275), consultada el 2026-10-25. Lo que la documentación no define se verificó contra las fechas del propio dataset o se marca como *pendiente de verificar*.

| Variable | Significado | Tipo | Unidad / valores |
|---|---|---|---|
| `instant` | Índice del registro | Entero (identificador) | Contador |
| `dteday` | Fecha | Fecha | Fecha calendario |
| `season` | Estación del año | Categórica | 1 = invierno, 2 = primavera, 3 = verano, 4 = otoño |
| `yr` | Año | Categórica | 0 = 2011, 1 = 2012 |
| `mnth` | Mes | Categórica | 1 a 12 |
| `hr` | Hora del día | Categórica | 0 a 23 |
| `holiday` | Indica si el día es festivo (según el calendario de feriados de Washington D.C.) | Binaria | 1 = festivo, 0 = no festivo (la documentación no detalla la codificación; verificado: los días con 1 en 2011 son festivos del calendario de Washington D. C., p. ej. 4 de julio y 15 de abril) |
| `weekday` | Día de la semana | Categórica | 0 = domingo a 6 = sábado (la documentación no lo indica; verificado contra `dteday`, p. ej. 2011-01-01 fue sábado y tiene 6) |
| `workingday` | Día laboral | Binaria | 1 si el día no es fin de semana ni festivo; 0 en otro caso |
| `weathersit` | Situación del clima | Categórica | 1 = despejado o pocas nubes; 2 = niebla con nubes; 3 = nieve o lluvia ligera (con tormenta o nubes dispersas); 4 = lluvia fuerte, granizo, tormenta o nieve con niebla |
| `temp` | Temperatura normalizada | Real | Celsius normalizado: (t − t_min)/(t_max − t_min), con t_min = −8 y t_max = +39 |
| `atemp` | Sensación térmica normalizada | Real | Celsius normalizado con t_min = −16 y t_max = +50 |
| `hum` | Humedad normalizada | Real | Valor dividido por 100 (máximo) |
| `windspeed` | Velocidad del viento normalizada | Real | Valor dividido por 67 (máximo); unidad original *pendiente de verificar* |
| `casual` | Alquileres de usuarios casuales | Entero (conteo) | Bicicletas alquiladas en esa hora |
| `registered` | Alquileres de usuarios registrados | Entero (conteo) | Bicicletas alquiladas en esa hora |
| `cnt` | Total de alquileres | Entero (conteo) | `casual` + `registered` |

### Variables que apoyan la pregunta

- **Demanda (lo que queremos describir):** `cnt`, `casual` y `registered`.
- **Franja horaria:** `hr`.
- **Tipo de día:** `workingday`, `holiday` y `weekday`.
- **Estación:** `season` (apoyada por `mnth`).
- **Clima:** `weathersit`, `temp`, `atemp`, `hum` y `windspeed`.
- **Periodo 2011–2012:** `yr` y `dteday`.

`instant` solo identifica el registro y no aporta al análisis.

> En esta entrega **no se transforma** el dataset: solo se revisa de forma descriptiva.

## 7. Resumen `.txt` generado desde el DataFrame

Con la clase `Reportes` creamos `EA2/resultados/resumen_dataframe.txt` (UTF-8, ruta relativa al notebook). El encabezado indica dataset, split, configuración, revisión y si se usó una muestra. La carpeta se crea si no existe y el archivo se regenera en cada ejecución.

In [ ]:
RUTA_RESUMEN = Path("../resultados/resumen_dataframe.txt")

# La carga fue completa si no se limitó max_rows ni se usó streaming
es_muestra = loader.max_rows is not None or loader.streaming
n_filas = f"{len(df):,}".replace(",", ".")
if es_muestra:
    muestra = f"sí, {n_filas} filas"
else:
    muestra = f"no, dataset completo ({n_filas} filas)"

encabezado = {
    "Dataset": loader.dataset_name,
    "Configuración": loader.config or "por defecto (sin configuración)",
    "Split": loader.split,
    "Revisión": loader.revision or "no fijada",
    "Muestra": muestra,
}

ruta = Reportes().guardar_reporte(df, RUTA_RESUMEN, encabezado=encabezado)
print(f"Archivo generado: {ruta}")

## 8. Verificación: leemos el archivo y lo imprimimos

Volvemos a abrir el `.txt` desde el disco para comprobar que su contenido coincide con el DataFrame.

In [ ]:
contenido = RUTA_RESUMEN.read_text(encoding="utf-8")
print(contenido)

In [ ]:
# Comprobaciones automáticas contra el DataFrame
assert f"Filas: {len(df)}" in contenido
assert f"Columnas: {len(df.columns)}" in contenido
assert all(columna in contenido for columna in df.columns)
print("El contenido del .txt coincide con el DataFrame.")

## 9. Conclusión

- Se obtuvo por código el dataset `t22000t/bike-sharing-tabular` (split `train`, revisión fija) y se cargó completo en un DataFrame.
- El DataFrame tiene una fila por hora de servicio y variables de demanda, calendario y clima que permiten responder la pregunta descriptiva.
- Se generó automáticamente `EA2/resultados/resumen_dataframe.txt` y se verificó leyéndolo de nuevo.
- La codificación de `weekday` y `holiday`, que UCI no documenta, se verificó contra las fechas del dataset. Queda pendiente de verificar la unidad original de `windspeed`.